### GQY consultancies.
This is an It consultancy company having three chatbots named as Goku (senior), Qwen and Yogu. You will porvide the problem to them and they will provide you the final solution after thoroughly discussing and evaluating each other suggestions.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from typing import Literal
from IPython.display import update_display,Markdown,display

In [ ]:
load_dotenv(override="true")
gemini_key = os.getenv("GOOGLE_API_KEY")

if not gemini_key:
    print("gemini key does not exist")

#### Below you can see i have configure ollama, but still using gemini models bcuz ollama is working slow.

In [ ]:
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/",api_key=gemini_key)
system_gemini_msg = """
You are an Principal Software Architect and IT consultant named as Goku in Company, who have to collabrate with other Two IT consultant named as qwen and yogu (which are also an AI).

flow:
1- initially you will provide solution to user problem.
2- which would then provided to qwen and yogu, who would evaluate your solution.
3- then if both qwen and yogu says "agreed"(lowercase) so your exact solution will be the final, else if qwen says "agreed" but yogu does not so yogu updated solution would provided to qwen, else if yogu says "agreed" but qwen does not so qwen updated solution would provided to yogu, else both are not agreed so both of them will discuss and their final solutions will be provided to you.
4- then if you completely agree with it then just say "agreed" else provide your updated solution.
5- then step2 till step4 will repeat again.

constraint:
Before actual answer-text must add your name "Goku" and answer in markdown. But when you have to just say "agreed" then dont add name or markdown format just plain "agreed" word. 
"""

ollama = OpenAI(base_url="http://localhost:11434/v1",api_key="ollama_key")
system_ollama_msg = """
You are an second Principal Software Architect and IT consultant named as Qwen in company, who have to collabrate with other Two IT consultant named as Goku and Yogu (which are also an AI).

flow:
1- initially Goku will provide solution to user problem.
2- which would then provided to you and yogu for evaluation.
3- then if you both qwen and yogu says "agreed"(lowercase) so Goku exact solution will be the final else if you says "agreed" but yogu does not so yogu updated solution would provided to you, else if yogu says "agreed" but you does not so your updated solution would provided to yogu, else both are not agreed so both of you will provide solutions to each other and discuss it, then the final solution will be provided to Goku. 
4- then if yogu disagree with goku and provide its own solution to you. So if you agree just say "agreed" else provide updated solution to yogu. then again yogu will evaluate your solution and say "agreed" else again provide its own. (this step is conditional and will happen recursively until both get agreed)
5- then if Goku agree with the final solution, so it will just say "agreed" else it provide updated solution.
6- then step2 till step5 will repeat again.

Before actual answer-text must add your name "Qwen" and answer in markdown. But when you have to just say "agreed" then dont add name or markdown format just plain "agreed" word.
"""

system_yogu_msg = """
You are an third Principal Software Architect and IT consultant named as Yogu in company, who have to collabrate with other Two IT consultant named as Goku and Qwen (which are also an AI).

flow:
1- initially Goku will provide solution to user problem.
2- which would then provided to you and qwen for evaluation.
3- then if you both yogu and qwen says "agreed"(lowercase) so Goku exact solution will be the final else if qwen says "agreed" but you does not so your updated solution would provided to qwen, else if you says "agreed" but qwen does not so qwen updated solution would provided to you, else both are not agreed so both of you will provide solutions to each other and discuss it, then the final solution will be provided to Goku. 
4- then if qwen disagree with goku and provide its own solution to you. So if you agree just say "agreed" else provide updated solution to qwen. then again qwen will evaluate your solution and say "agreed" else again provide its own. (this step is conditional and will happen recursively until both get agreed)
5- then if Goku agree with it, so it will just say "agreed" else it provide updated solution.
6- then step2 till step5 will repeat again.

Before actual answer-text must add your name "Yogu" and answer in markdown. But when you have to just say "agreed" then dont add name or markdown format just plain "agreed" word.
"""

In [ ]:
chat_history = []

In [ ]:
def update_chat_history(content:str, role:Literal["user","assistant","system"]):
        chat_history.append({
            "role":role,
            "content":content
        })

In [ ]:
goku_msgs = [
    {
        "role":"system",
        "content":system_gemini_msg
    }
]
qwen_msgs = [
        {
        "role":"system",
        "content":system_ollama_msg
    }
]
yogu_msgs = [
        {
        "role":"system",
        "content":system_yogu_msg
    }
]

In [ ]:
def update_goku_msgs(content:str, role:Literal["user","assistant","system"]):
        goku_msgs.append({
            "role":role,
            "content":content
        })

def update_qwen_msgs(content:str, role:Literal["user","assistant","system"]):
        qwen_msgs.append({
            "role":role,
            "content":content
        })

def update_yogu_msgs(content:str, role:Literal["user","assistant","system"]):
        yogu_msgs.append({
            "role":role,
            "content":content
        })

In [ ]:


def consultants(query:str):
    initial_res = None
    if(query):
        update_chat_history(query,"user")
        # goku_msgs += chat_history
        goku_msgs.append({
            "role":"user",
            "content":query
        })
        handle_display = display(Markdown(""),display_id=True)
        initial_res = gemini.chat.completions.create(model="gemini-3.1-flash-lite",messages=goku_msgs, stream="true")
        streamed_result = ""
        for chunk in initial_res:
            streamed_result += chunk.choices[0].delta.content or ''
            update_display(Markdown(streamed_result),display_id=handle_display.display_id)
        update_goku_msgs(streamed_result,"assistant")
        update_chat_history(streamed_result,"assistant")
    else:
        print("provide problem as query")
        return
    
    isStop = False
    isStop_qwen_yogu = False

    while not isStop:
        handle_display = display(Markdown(""),display_id=True)

        qwen_msgs.append({
            "role":"user",
            "content":goku_msgs[-1]["content"]
        })

        yogu_msgs.append({
            "role":"user",
            "content":goku_msgs[-1]["content"]
        })

        qwen_streamed_res = gemini.chat.completions.create(
            model="gemini-3.5-flash-lite",
            messages=qwen_msgs,
            stream=True
            )
        qwen_streamed_result = ""
        for chunk in qwen_streamed_res:
                qwen_streamed_result += chunk.choices[0].delta.content or ''
                update_display(Markdown(qwen_streamed_result),display_id=handle_display.display_id)
        update_qwen_msgs(
                qwen_streamed_result,
                "assistant"
            )
        update_chat_history(qwen_streamed_result,"assistant")

        yogu_streamed_res = gemini.chat.completions.create(
            model="gemini-3.5-flash-lite",
            messages=yogu_msgs,
            stream=True
            )
        yogu_streamed_result = ""
        for chunk in yogu_streamed_res:
            yogu_streamed_result += chunk.choices[0].delta.content or ''
        update_display(Markdown(yogu_streamed_result),display_id=handle_display.display_id)
        update_yogu_msgs(
            yogu_streamed_result,
            "assistant"
        )
        update_chat_history(yogu_streamed_result,"assistant")

        qwen_yogu_final_solution = ""

        if (qwen_streamed_result or "").strip(' "\'`') == "agreed" and (yogu_streamed_result or "").strip(' "\'`') == "agreed":
            isStop = True
            break
        else:
            while not isStop_qwen_yogu:

                if qwen_msgs[-1]["content"].strip(' "\'`') != "agreed":
                    qwen_msgs.append({
                        "role":"user",
                        "content":"As you are not agree with previous, so provide your updated solution."
                    })
                    qwen_streamed_res = gemini.chat.completions.create(
                    model="gemini-3.5-flash-lite",
                    messages=qwen_msgs,
                    stream=True
                    )
                    qwen_streamed_result = ""
                    for chunk in qwen_streamed_res:
                        qwen_streamed_result += chunk.choices[0].delta.content or ''
                        update_display(Markdown(qwen_streamed_result),display_id=handle_display.display_id)

                    update_qwen_msgs(
                        qwen_streamed_result,
                        "assistant"
                    )
                    update_chat_history(qwen_streamed_result,"assistant")
            
                    yogu_msgs.append({
                        "role":"user",
                        "content":qwen_msgs[-1]["content"]
                    })
                    yogu_streamed_res = gemini.chat.completions.create(
                    model="gemini-3.5-flash-lite",
                    messages=yogu_msgs,
                    stream=True
                    )
                    yogu_streamed_result = ""
                    for chunk in yogu_streamed_res:
                        yogu_streamed_result += chunk.choices[0].delta.content or ''
                        update_display(Markdown(yogu_streamed_result),display_id=handle_display.display_id)
                        if (chunk.choices[0].delta.content or "").strip(' "\'`') == "agreed":
                                qwen_yogu_final_solution += qwen_msgs[-1]["content"]
                                isStop_qwen_yogu = True
                                break
                    update_yogu_msgs(
                        yogu_streamed_result,
                        "assistant"
                    )
                    update_chat_history(yogu_streamed_result,"assistant")

                elif yogu_msgs[-1]["content"].strip(' "\'`') != "agreed":
                    yogu_msgs.append({
                        "role":"user",
                        "content":"As you are not agree with previous, so provide your updated solution."
                    })
                    yogu_streamed_res = gemini.chat.completions.create(
                    model="gemini-3.5-flash-lite",
                    messages=yogu_msgs,
                    stream=True
                    )
                    yogu_streamed_result = ""
                    for chunk in yogu_streamed_res:
                        yogu_streamed_result += chunk.choices[0].delta.content or ''
                        update_display(Markdown(yogu_streamed_result),display_id=handle_display.display_id)

                    update_yogu_msgs(
                        yogu_streamed_result,
                        "assistant"
                    )
                    update_chat_history(yogu_streamed_result,"assistant")
            
                    qwen_msgs.append({
                        "role":"user",
                        "content":yogu_msgs[-1]["content"]
                    })
                    qwen_streamed_res = gemini.chat.completions.create(
                    model="gemini-3.5-flash-lite",
                    messages=qwen_msgs,
                    stream=True
                    )
                    qwen_streamed_result = ""
                    for chunk in qwen_streamed_res:
                        qwen_streamed_result += chunk.choices[0].delta.content or ''
                        update_display(Markdown(qwen_streamed_result),display_id=handle_display.display_id)
                        if (chunk.choices[0].delta.content or "").strip(' "\'`') == "agreed":
                                qwen_yogu_final_solution += yogu_msgs[-1]["content"]
                                isStop_qwen_yogu = True
                                break
                    update_qwen_msgs(
                        qwen_streamed_result,
                        "assistant"
                    )
                    update_chat_history(qwen_streamed_result,"assistant")

        goku_msgs.append({
            "role":"user",
            "content":qwen_yogu_final_solution
        })
        goku_streamed_res = gemini.chat.completions.create(
            model="gemini-3.1-flash-lite",
            messages=goku_msgs,
            stream=True
            )
        goku_streamed_result = ""
        for chunk in goku_streamed_res:
            goku_streamed_result += chunk.choices[0].delta.content or ''
            update_display(Markdown(goku_streamed_result),display_id=handle_display.display_id)
            if (chunk.choices[0].delta.content or "").strip(' "\'`') == "agreed":
                    isStop = True
                    break
        update_goku_msgs(
            goku_streamed_result,
            "assistant"
        )
        update_chat_history(goku_streamed_result,"assistant")
 
goku_msgs = [
    {
        "role":"system",
        "content":system_gemini_msg
    }
]
qwen_msgs = [
        {
        "role":"system",
        "content":system_ollama_msg
    }
]
yogu_msgs = [
        {
        "role":"system",
        "content":system_yogu_msg
    }
]
chat_history = []

In [ ]:
consultants("""
I run a home nursing care business, how the IT services can boost up my business?
""")